In [1]:
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import hdbscan

/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
node_labels = [
    'AndJunction', 'ApplicationCollaboration', 'ApplicationComponent',
    'ApplicationEvent', 'ApplicationFunction', 'ApplicationInteraction',
    'ApplicationInterface', 'ApplicationProcess', 'ApplicationService',
    'Artifact', 'Assessment', 'BusinessActor', 'BusinessCollaboration',
    'BusinessEvent', 'BusinessFunction', 'BusinessInteraction',
    'BusinessInterface', 'BusinessObject', 'BusinessProcess', 'BusinessRole',
    'BusinessService', 'Capability', 'CommunicationNetwork', 'Constraint',
    'Contract', 'CourseOfAction', 'DataObject', 'Device', 'Driver',
    'Equipment', 'Facility', 'Goal', 'Grouping', 'ImplementationEvent',
    'Junction', 'Location', 'Meaning', 'Node', 'OrJunction', 'Outcome',
    'Path', 'Plateau', 'Principle', 'Product', 'Representation',
    'Requirement', 'Resource', 'Stakeholder', 'SystemSoftware',
    'TechnologyFunction', 'TechnologyInterface', 'TechnologyProcess',
    'TechnologyService', 'Value', 'ValueStream', 'WorkPackage'
]

edge_labels = [
    'Access', 'Aggregation', 'Assignment', 'Association',
    'Composition', 'Flow', 'Influence', 'Realization',
    'Serving', 'Specialization', 'Triggering'
]

In [3]:
print("Loading model...", end="")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("DONE")

Loading model...DONE


In [5]:
print("Generating embeddings for node labels...", end="")
embeddings_node = model.encode(node_labels, show_progress_bar=True)
embeddings_node = normalize(embeddings_node)
print("DONE")

print("Generating embeddings for edge labels...", end="")
embeddings_edge = model.encode(edge_labels, show_progress_bar=True)
embeddings_edge = normalize(embeddings_edge)
print("DONE")

Generating embeddings for node labels...

Batches: 100%|██████████| 2/2 [00:00<00:00, 27.24it/s]


DONE
Generating embeddings for edge labels...

Batches: 100%|██████████| 1/1 [00:00<00:00, 131.10it/s]

DONE


In [7]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric="euclidean")

print("\nClustering node labels...", end="")
labels_nodes = clusterer.fit_predict(embeddings_node)

print("DONE")

print("\nClustering edge labels...", end="")
labels_edges = clusterer.fit_predict(embeddings_edge)

print("DONE")


Clustering node labels...DONE

Clustering edge labels...DONE


In [9]:
clusters_nodes = defaultdict(list)
for word, label in zip(node_labels, labels_nodes):
    cluster_name = f"Cluster {label}" if label != -1 else "Outlier (no cluster)"
    clusters_nodes[cluster_name].append(word)

clusters_edges = defaultdict(list)
for word, label in zip(edge_labels, labels_edges):
    cluster_name = f"Cluster {label}" if label != -1 else "Outlier (no cluster)"
    clusters_edges[cluster_name].append(word)

In [10]:
print("\n--- Semantic Clusters ---")
print("\nNode Labels:")
for cluster_name, words in clusters_nodes.items():
    print(f"\n{cluster_name}:")
    for word in words:
        print(f"  - {word}")

print("\nEdge Labels:")
for cluster_name, words in clusters_edges.items():
    print(f"\n{cluster_name}:")
    for word in words:
        print(f"  - {word}")


--- Semantic Clusters ---

Node Labels:

Outlier (no cluster):
  - AndJunction
  - Artifact
  - Assessment
  - CommunicationNetwork
  - Constraint
  - Contract
  - CourseOfAction
  - DataObject
  - Driver
  - Facility
  - Goal
  - Grouping
  - Junction
  - Meaning
  - Node
  - OrJunction
  - Outcome
  - Plateau
  - Principle
  - Representation
  - Resource
  - Stakeholder
  - Value
  - ValueStream
  - WorkPackage

Cluster 0:
  - ApplicationCollaboration
  - ApplicationComponent
  - ApplicationEvent
  - ApplicationFunction
  - ApplicationInteraction
  - ApplicationInterface
  - ApplicationProcess
  - ApplicationService
  - BusinessActor
  - BusinessCollaboration
  - BusinessEvent
  - BusinessFunction
  - BusinessInteraction
  - BusinessInterface
  - BusinessObject
  - BusinessProcess
  - BusinessRole
  - BusinessService
  - ImplementationEvent
  - SystemSoftware
  - TechnologyFunction
  - TechnologyInterface
  - TechnologyProcess
  - TechnologyService

Cluster 1:
  - Capability
  - Dev

In [12]:
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
import umap
import hdbscan

# --- 1. Array di parole ---
words = [
    'AndJunction', 'ApplicationCollaboration', 'ApplicationComponent',
    'ApplicationEvent', 'ApplicationFunction', 'ApplicationInteraction',
    'ApplicationInterface', 'ApplicationProcess', 'ApplicationService',
    'Artifact', 'Assessment', 'BusinessActor', 'BusinessCollaboration',
    'BusinessEvent', 'BusinessFunction', 'BusinessInteraction',
    'BusinessInterface', 'BusinessObject', 'BusinessProcess', 'BusinessRole',
    'BusinessService', 'Capability', 'CommunicationNetwork', 'Constraint',
    'Contract', 'CourseOfAction', 'DataObject', 'Device', 'Driver',
    'Equipment', 'Facility', 'Goal', 'Grouping', 'ImplementationEvent',
    'Junction', 'Location', 'Meaning', 'Node', 'OrJunction', 'Outcome',
    'Path', 'Plateau', 'Principle', 'Product', 'Representation',
    'Requirement', 'Resource', 'Stakeholder', 'SystemSoftware',
    'TechnologyFunction', 'TechnologyInterface', 'TechnologyProcess',
    'TechnologyService', 'Value', 'ValueStream', 'WorkPackage'
]

# --- 2. Embedding ---
print("Carico il modello...")
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(words, show_progress_bar=True)
embeddings = normalize(embeddings)

# ============================================================
# METODO A — K-Means con Silhouette Score automatico
# ============================================================
print("\n=== METODO A: Silhouette Score ===")

best_k, best_score, best_labels = None, -1, None

for k in range(2, min(20, len(words))):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels)
    print(f"  k={k:2d} → silhouette={score:.4f}")
    if score > best_score:
        best_score, best_k, best_labels = score, k, labels

print(f"\n✅ Numero di cluster ottimale: {best_k} (silhouette={best_score:.4f})")

clusters_a = defaultdict(list)
for word, label in zip(words, best_labels):
    clusters_a[f"Cluster {label}"].append(word)
print("\nRisultati Metodo A:")
for cluster, members in sorted(clusters_a.items()):
    print(f"  {cluster}: {', '.join(members)}")

# ============================================================
# METODO B — UMAP (riduzione 384D → 5D) + HDBSCAN
# ============================================================
print("\n=== METODO B: UMAP + HDBSCAN ===")

reducer = umap.UMAP(n_components=5, n_neighbors=8, min_dist=0.1, random_state=42)
reduced = reducer.fit_transform(embeddings)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=3,   # abbassa per trovare più cluster piccoli
    min_samples=1,        # più sensibile alle strutture locali
    metric="euclidean"
)
labels_b = clusterer.fit_predict(reduced)

n_clusters_b = len(set(labels_b)) - (1 if -1 in labels_b else 0)
n_outliers_b = list(labels_b).count(-1)
print(f"✅ Cluster trovati: {n_clusters_b} | Outlier: {n_outliers_b}")

clusters_b = defaultdict(list)
for word, label in zip(words, labels_b):
    name = f"Cluster {label}" if label != -1 else "⚠️ Outlier"
    clusters_b[name].append(word)
print("\nRisultati Metodo B:")
for cluster, members in sorted(clusters_b.items()):
    print(f"  {cluster}: {', '.join(members)}")

# ============================================================
# METODO C — Agglomerative Clustering con distance_threshold
# (nessun k da scegliere, taglia l'albero in automatico)
# ============================================================
print("\n=== METODO C: Agglomerative + distance_threshold ===")

agg = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1.0,  # abbassa per più cluster, alza per meno
    linkage="ward"
)
labels_c = agg.fit_predict(embeddings)
n_clusters_c = len(set(labels_c))
print(f"✅ Cluster trovati: {n_clusters_c}")

clusters_c = defaultdict(list)
for word, label in zip(words, labels_c):
    clusters_c[f"Cluster {label}"].append(word)
print("\nRisultati Metodo C:")
for cluster, members in sorted(clusters_c.items()):
    print(f"  {cluster}: {', '.join(members)}")

Carico il modello...


Batches: 100%|██████████| 2/2 [00:00<00:00, 55.34it/s]
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.p


=== METODO A: Silhouette Score ===
  k= 2 → silhouette=0.0866
  k= 3 → silhouette=0.0800
  k= 4 → silhouette=0.0842


/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in mat

  k= 5 → silhouette=0.0916
  k= 6 → silhouette=0.0848
  k= 7 → silhouette=0.0956
  k= 8 → silhouette=0.0903
  k= 9 → silhouette=0.0928
  k=10 → silhouette=0.1027
  k=11 → silhouette=0.0925
  k=12 → silhouette=0.0836


/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/simone/Projects/university/CMiner/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in mat

  k=13 → silhouette=0.0735
  k=14 → silhouette=0.0928
  k=15 → silhouette=0.0829
  k=16 → silhouette=0.0908
  k=17 → silhouette=0.0951
  k=18 → silhouette=0.0980
  k=19 → silhouette=0.0732

✅ Numero di cluster ottimale: 10 (silhouette=0.1027)

Risultati Metodo A:
  Cluster 0: Artifact, Plateau
  Cluster 1: CommunicationNetwork, Facility, SystemSoftware, TechnologyFunction, TechnologyInterface, TechnologyProcess, TechnologyService, WorkPackage
  Cluster 2: Device, Driver, Equipment, Goal, Grouping, Location, Meaning, Outcome, Product, Representation, Value
  Cluster 3: ApplicationCollaboration, ApplicationComponent, ApplicationEvent, ApplicationFunction, ApplicationInteraction, ApplicationInterface, ApplicationProcess, ApplicationService, ImplementationEvent
  Cluster 4: BusinessActor, BusinessCollaboration, BusinessEvent, BusinessFunction, BusinessInteraction, BusinessInterface, BusinessObject, BusinessProcess, BusinessRole, BusinessService
  Cluster 5: AndJunction, OrJunction
  Cluste